# EXP-003 — affordance memory explorer

Conservative generalization experiment. Preserve EXP-001 visible-pixel random exploration while adding a small memory of object/action effects.

Validation gate before submission: local score >= 0.2123845862 and levels >= 7/183.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, random, zlib
from pathlib import Path
from collections import defaultdict, deque
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

EXP_ID='EXP-003'
MAX_MOVES=1000
SEED=42
USE_PER_GAME_SEED=False
MEMORY_BIAS_PROB=0.15  # conservative: keep 85% EXP-001 random behavior
WORK=Path('/kaggle/working')
ART=WORK/'exp003_artifacts'
ART.mkdir(exist_ok=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    get_ipython().system('curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games')
    mode='online'; envdir=''
else:
    mode='offline'; envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'

(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE={mode}\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)

def rng_for(g):
    return random.Random(SEED if not USE_PER_GAME_SEED else SEED+(zlib.adler32(g.encode())&0xffffffff))

def get_frame(obs):
    if obs is None or not getattr(obs,'frame',None): return None
    return np.asarray(obs.frame[-1])

def frame_hash(frame):
    if frame is None: return None
    return int(zlib.adler32(np.asarray(frame,dtype=np.uint8).tobytes()) & 0xffffffff)

def diff_summary(a,b):
    if a is None or b is None or a.shape!=b.shape:
        return {'frame_changed':None,'diff_pixels':None,'diff_cx':None,'diff_cy':None}
    d=a!=b; ys,xs=np.where(d)
    return {'frame_changed':bool(len(xs)>0),'diff_pixels':int(len(xs)),'diff_cx':float(xs.mean()) if len(xs) else None,'diff_cy':float(ys.mean()) if len(ys) else None}

def random_visible_pixel(frame,rng):
    color=rng.choice(np.unique(frame).tolist())
    ys,xs=np.where(frame==color)
    i=rng.randint(0,len(xs)-1)
    return {'x':int(xs[i]),'y':int(ys[i])}

def connected_components(frame,min_area=2,max_area=400):
    arr=np.asarray(frame); h,w=arr.shape[:2]
    vals,cnts=np.unique(arr,return_counts=True); counts=dict(zip(vals.tolist(),cnts.tolist()))
    ignore=set(c for c,n in counts.items() if n>0.30*h*w)
    ignore |= set(c for c,n in sorted(counts.items(),key=lambda kv:kv[1],reverse=True)[:2])
    seen=np.zeros((h,w),bool); comps=[]
    for y in range(h):
        for x in range(w):
            if seen[y,x]: continue
            color=int(arr[y,x])
            if color in ignore:
                seen[y,x]=True; continue
            q=deque([(x,y)]); seen[y,x]=True; pts=[]
            while q:
                cx,cy=q.popleft(); pts.append((cx,cy))
                for nx,ny in ((cx+1,cy),(cx-1,cy),(cx,cy+1),(cx,cy-1)):
                    if 0<=nx<w and 0<=ny<h and not seen[ny,nx] and int(arr[ny,nx])==color:
                        seen[ny,nx]=True; q.append((nx,ny))
            area=len(pts)
            if min_area<=area<=max_area:
                xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
                comps.append({'color':color,'area':int(area),'x0':int(min(xs)),'x1':int(max(xs)),'y0':int(min(ys)),'y1':int(max(ys)),'cx':float(sum(xs)/area),'cy':float(sum(ys)/area),'w':int(max(xs)-min(xs)+1),'h':int(max(ys)-min(ys)+1),'touches_edge':bool(min(xs)==0 or min(ys)==0 or max(xs)==w-1 or max(ys)==h-1)})
    return comps

def component_candidates(frame, memory):
    comps=connected_components(frame)
    ranked=[]
    for c in comps:
        if c['cy']>61 or c['area']>180 or c['w']>28 or c['h']>28: continue
        key=f"{round(c['cx'])}:{round(c['cy'])}:{c['color']}"
        stats=memory['object_stats'].get(key,{})
        trials=stats.get('trials',0); useful=stats.get('useful',0); noops=stats.get('noops',0)
        score=0.0
        score += 1.0/(1.0+abs(c['area']-12))
        score += 0.20 if not c['touches_edge'] else -0.20
        # soft affordance prior: prefer components that changed frames before, but keep novelty.
        score += 0.50*useful - 0.20*noops - 0.03*trials
        ranked.append((score,key,c))
    ranked.sort(key=lambda x:x[0],reverse=True)
    return ranked

def choose_memory_biased_action(frame, valid_actions, rng, memory):
    complex_actions=[a for a in valid_actions if a.is_complex()]
    if not complex_actions or frame is None: return None
    ranked=component_candidates(frame,memory)
    if not ranked: return None
    score,key,c=ranked[0]
    a=rng.choice(complex_actions)
    data={'x':int(round(c['cx'])),'y':int(round(c['cy']))}
    reasoning={'exp_id':EXP_ID,'policy':'memory_biased_component_click','candidate_key':key,'target_color':c['color'],'target_area':c['area'],'target_cx':c['cx'],'target_cy':c['cy'],'candidate_score':score}
    return a,data,reasoning,key

def choose_exp001_action(frame, valid_actions, rng):
    a=rng.choice(valid_actions)
    data=random_visible_pixel(frame,rng) if a.is_complex() and frame is not None else {}
    return a,data,{'exp_id':EXP_ID,'policy':'exp001_random_visible_pixel_fallback'},None

def play(env,g):
    rng=rng_for(g)
    r=env._last_response
    last=None; last_policy=None
    action_counts=defaultdict(int); policy_counts=defaultdict(int); noop_counts=defaultdict(int)
    memory={'object_stats':{},'effects':[]}
    valid_base=[a for a in GameAction if a is not GameAction.RESET]
    for m in range(1,MAX_MOVES+1):
        if r is None or r.state==GameState.WIN: break
        if r.state in (GameState.GAME_OVER,GameState.NOT_PLAYED):
            r=env.step(GameAction.RESET,{})
            last='RESET'; last_policy='reset'; action_counts['RESET']+=1; policy_counts['reset']+=1
            continue
        frame=get_frame(r)
        valid=list(getattr(env,'action_space',[])) or valid_base
        valid=[a for a in valid if a is not GameAction.RESET]
        # Keep EXP-001 semantics: if env action_space is incomplete/strict, use it; otherwise fallback to enum.
        if not valid: valid=valid_base
        chosen=None; candidate_key=None
        if rng.random()<MEMORY_BIAS_PROB:
            chosen=choose_memory_biased_action(frame,valid,rng,memory)
        if chosen is None:
            a,data,reasoning,candidate_key=choose_exp001_action(frame,valid,rng)
        else:
            a,data,reasoning,candidate_key=chosen
        prev_frame=frame.copy() if frame is not None else None
        prev_levels=int(r.levels_completed) if r else None
        prev_state=r.state.name if r else None
        r=env.step(a,data=data,reasoning=reasoning)
        next_frame=get_frame(r)
        eff=diff_summary(prev_frame,next_frame)
        eff.update({'step':m,'action':a.name,'policy':reasoning.get('policy'),'data':data,'candidate_key':candidate_key,'levels_before':prev_levels,'levels_after':int(r.levels_completed) if r else None,'level_delta':(int(r.levels_completed)-prev_levels) if r and prev_levels is not None else None,'state_before':prev_state,'state_after':r.state.name if r else None})
        useful=bool(eff.get('frame_changed')) or bool(eff.get('level_delta',0)) or (eff.get('state_after')!=eff.get('state_before'))
        if candidate_key:
            st=memory['object_stats'].setdefault(candidate_key,{'trials':0,'useful':0,'noops':0,'total_diff_pixels':0})
            st['trials']+=1
            if useful: st['useful']+=1
            if eff.get('frame_changed') is False: st['noops']+=1
            st['total_diff_pixels']+=int(eff.get('diff_pixels') or 0)
        if eff.get('frame_changed') is False:
            noop_counts[a.name]+=1
        memory['effects'].append(eff)
        last=a.name; last_policy=reasoning.get('policy')
        action_counts[a.name]+=1; policy_counts[last_policy]+=1
    return {'game_id':g,'moves':m,'state':r.state.name if r else 'unknown','levels_completed':int(r.levels_completed) if r else 0,'last_action':last,'last_policy':last_policy,'action_counts':dict(action_counts),'policy_counts':dict(policy_counts),'noop_counts':dict(noop_counts),'object_stats':memory['object_stats'],'effects_tail':memory['effects'][-50:]}

arcade=arc_agi.Arcade()
infos=list(arcade.available_environments)
rows=[]; details=[]
print(EXP_ID,'envs=',len(infos),'MAX_MOVES=',MAX_MOVES,'SEED=',SEED,'MEMORY_BIAS_PROB=',MEMORY_BIAS_PROB)
for i,info in enumerate(infos,1):
    row=play(arcade.make(info.game_id),info.game_id)
    details.append(row)
    flat={k:v for k,v in row.items() if k not in ('action_counts','policy_counts','noop_counts','object_stats','effects_tail')}
    rows.append(flat)
    print(f'[{i:03d}/{len(infos):03d}] {flat}')
pd.DataFrame(rows).to_csv(ART/'exp003_run_results.csv',index=False)
(ART/'exp003_run_details.json').write_text(json.dumps(details,indent=2))

policy_totals=defaultdict(int); action_totals=defaultdict(int); noop_totals=defaultdict(int); object_memory={}
for d in details:
    for k,v in d['policy_counts'].items(): policy_totals[k]+=int(v)
    for k,v in d['action_counts'].items(): action_totals[k]+=int(v)
    for k,v in d['noop_counts'].items(): noop_totals[k]+=int(v)
    object_memory[d['game_id']]=d.get('object_stats',{})
(ART/'exp003_policy_counts.json').write_text(json.dumps(dict(policy_totals),indent=2,sort_keys=True))
(ART/'exp003_action_counts.json').write_text(json.dumps(dict(action_totals),indent=2,sort_keys=True))
(ART/'exp003_noop_counts.json').write_text(json.dumps(dict(noop_totals),indent=2,sort_keys=True))
(ART/'exp003_effect_memory.json').write_text(json.dumps(object_memory,indent=2,sort_keys=True))

summary={'exp_id':EXP_ID,'max_moves':MAX_MOVES,'seed':SEED,'use_per_game_seed':USE_PER_GAME_SEED,'memory_bias_prob':MEMORY_BIAS_PROB,'change':'EXP-001 fallback plus conservative object/action/effect memory bias'}
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sc=arcade.get_scorecard()
    summary.update(score=float(sc.score),total_environments_completed=int(sc.total_environments_completed),total_environments=int(sc.total_environments),total_levels_completed=int(sc.total_levels_completed),total_levels=int(sc.total_levels),total_actions=int(sc.total_actions))
    pd.DataFrame([{'game':e.id,'score':float(e.score),'levels_completed':int(e.levels_completed),'actions':int(e.actions),'completed':bool(e.completed)} for e in sc.environments]).to_csv(ART/'exp003_scorecard_by_environment.csv',index=False)
    pd.DataFrame([{'tag':t.id,'score':float(t.score),'levels_completed':int(t.levels_completed),'number_of_environments':int(t.number_of_environments)} for t in (sc.tags_scores or [])]).to_csv(ART/'exp003_scorecard_by_tag.csv',index=False)
    print('Score:', f'{sc.score:.4f}', 'Levels:', f'{sc.total_levels_completed}/{sc.total_levels}', 'Actions:', sc.total_actions)
(ART/'exp003_scorecard_summary.json').write_text(json.dumps(summary,indent=2))
sp=WORK/'submission.parquet'
if not sp.exists():
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
manifest=sorted(str(p) for p in ART.glob('*'))
pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('submission:', sp, sp.exists())
print('artifacts:', ART)
print(json.dumps(summary,indent=2))
summary
